# Step 3 — Windowed Soft-Argmax
DINOv2 (finetuned) · grid search K × temperature · comparison vs argmax · SPair-71k

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
RESULTS_DIR = f'{DRIVE_ROOT}/results/step3'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted.')

In [ ]:
# Cell 1 — Install + clone
!pip install -q timm pandas
import os, subprocess, sys
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/YOUR_USER/semantic_correspondence.git',
                    REPO_PATH], check=False)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('Ready.')

In [ ]:
# Cell 2 — Imports + config
import torch, numpy as np, json, time, os
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SPAIR_BASE    = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
PAIR_ANN_PATH = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH   = f'{SPAIR_BASE}/Layout'
IMAGE_PATH    = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE  = 'large'
PCK_ALPHA     = 0.1
THRESHOLDS    = [0.05, 0.1, 0.2]
DINOV2_W      = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
FINETUNED_W   = f'{DRIVE_ROOT}/weights/finetuned/dinov2_best.pth'
IMG_SIZE      = 518
PATCH_SIZE    = 14

In [ ]:
# Cell 3 — Inline utility functions
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.datatype = datatype
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': ann['trg_bndbox'], 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

def extract_dense_features(model, img_tensor):
    with torch.no_grad():
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

def find_best_match_windowed_softargmax(similarities, width, height, K=5, temperature=1.0):
    """Windowed soft-argmax: find argmax patch, then soft-argmax in K×K window."""
    idx = similarities.argmax().item()
    cx, cy = idx % width, idx // width

    x0 = max(cx - K // 2, 0)
    x1 = min(cx + K // 2 + 1, width)
    y0 = max(cy - K // 2, 0)
    y1 = min(cy + K // 2 + 1, height)

    window_sims = similarities.reshape(height, width)[y0:y1, x0:x1]
    weights = torch.softmax(window_sims.flatten() * temperature, dim=0)

    ys, xs = torch.meshgrid(
        torch.arange(y0, y1, dtype=torch.float32, device=similarities.device),
        torch.arange(x0, x1, dtype=torch.float32, device=similarities.device),
        indexing='ij'
    )
    soft_x = (weights * xs.flatten()).sum().item()
    soft_y = (weights * ys.flatten()).sum().item()
    return soft_x, soft_y

def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

def evaluate_with_softargmax(model, dataset, device, thresholds, patch_size,
                               resized_size, K=5, temperature=1.0):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_windowed_softargmax(sim, W, H, K=K, temperature=temperature)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved -> {path}')
    return stats

print('Utility functions loaded.')

In [ ]:
# Cell 4 — Load DINOv2 (finetuned if available, else pretrained)
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base

dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)

if os.path.exists(FINETUNED_W):
    ckpt = torch.load(FINETUNED_W, map_location=device)
    dinov2.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f'Loaded finetuned weights from {FINETUNED_W}')
else:
    ckpt = torch.load(DINOV2_W, map_location=device)
    dinov2.load_state_dict(ckpt, strict=True)
    print(f'Loaded pretrained weights (finetuned not found).')

dinov2.eval()

test_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
print(f'Test set: {len(test_ds)} pairs')

In [ ]:
# Cell 5 — Baseline: argmax (for comparison)
print('=== Baseline: argmax ===')

def evaluate_with_argmax(model, dataset, device, thresholds, patch_size, resized_size):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

res_base = evaluate_with_argmax(dinov2, test_ds, device, THRESHOLDS, PATCH_SIZE, IMG_SIZE)
stats_base = save_exp_results(res_base, 'exp3_argmax_baseline', THRESHOLDS, RESULTS_DIR)

In [ ]:
# Cell 6 — Grid search: K × temperature
K_values = [3, 5, 7, 9]
T_values = [0.5, 1.0, 2.0, 5.0, 10.0]

grid_results = []
best_pck_grid = -1.0
best_K, best_T = K_values[0], T_values[0]

for K in K_values:
    for T in T_values:
        print(f'--- K={K}, T={T} ---')
        res = evaluate_with_softargmax(dinov2, test_ds, device, [0.1], PATCH_SIZE, IMG_SIZE, K=K, temperature=T)
        mean_pck = float(np.mean([m['pck_scores'][0.1] for m in res]))
        grid_results.append({'K': K, 'T': T, 'pck@0.10': mean_pck})
        print(f'  PCK@0.10={mean_pck:.2f}%')
        if mean_pck > best_pck_grid:
            best_pck_grid = mean_pck
            best_K, best_T = K, T

print(f'\nBest: K={best_K}, T={best_T} -> PCK@0.10={best_pck_grid:.2f}%')

# Save grid search results
df_grid = pd.DataFrame(grid_results)
df_grid.to_csv(f'{RESULTS_DIR}/grid_search_KxT.csv', index=False)
print('Grid saved.')

In [ ]:
# Cell 7 — Full evaluation with best K and T
print(f'=== Full eval: windowed soft-argmax (K={best_K}, T={best_T}) ===')
res_softmax = evaluate_with_softargmax(
    dinov2, test_ds, device, THRESHOLDS, PATCH_SIZE, IMG_SIZE, K=best_K, temperature=best_T
)
stats_softmax = save_exp_results(
    res_softmax, f'exp3_softargmax_K{best_K}_T{best_T}', THRESHOLDS, RESULTS_DIR
)

In [ ]:
# Cell 8 — Summary table
pivot = pd.DataFrame(grid_results).pivot(index='K', columns='T', values='pck@0.10')
print('\n=== Grid Search PCK@0.10 (%) ===')
print(pivot.to_string())

rows = [
    {'Method': 'Argmax (baseline)', 'PCK@0.05': f"{stats_base.get('pck@0.05',{}).get('mean',0):.2f}%",
     'PCK@0.10': f"{stats_base.get('pck@0.10',{}).get('mean',0):.2f}%",
     'PCK@0.20': f"{stats_base.get('pck@0.20',{}).get('mean',0):.2f}%"},
    {'Method': f'Soft-argmax K={best_K} T={best_T}',
     'PCK@0.05': f"{stats_softmax.get('pck@0.05',{}).get('mean',0):.2f}%",
     'PCK@0.10': f"{stats_softmax.get('pck@0.10',{}).get('mean',0):.2f}%",
     'PCK@0.20': f"{stats_softmax.get('pck@0.20',{}).get('mean',0):.2f}%"},
]
df_summary = pd.DataFrame(rows)
print('\n=== Step 3 Summary ===')
print(df_summary.to_string(index=False))
df_summary.to_csv(f'{RESULTS_DIR}/step3_summary.csv', index=False)
print(f'Saved to {RESULTS_DIR}/step3_summary.csv')